In [3]:
import torch

In [4]:
# Safe loading for trusted models (option 1)
# checkpoint = torch.load(r"C:\Users\gargp\Desktop\NewDS\checkpoints\best_model_dim8.pt", weights_only=False)  # Only if you trust the source

# Secure loading with allowlisting (option 2)
from torch.serialization import add_safe_globals
import numpy as np

# Allowlist numpy scalar type
add_safe_globals([np.core.multiarray.scalar])

# Now load safely
checkpoint = torch.load(r"C:\Users\gargp\Desktop\NewDS\checkpoints\best_model_dim8.pt", weights_only=False)


In [9]:
print("Checkpoint keys:", checkpoint.keys())


Checkpoint keys: dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_metrics', 'hidden_dim'])


In [10]:
# Initialize model with saved parameters
from models.fingat import  FinGAT
model = FinGAT(
    input_dim=16,  # Must match original config
    hidden_dim=checkpoint['hidden_dim'],
    embed_dim=8,  # From your config
    num_stocks=455,
    num_sectors=19  # Adjust based on your data
)

# Load weights
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()  # Set to evaluation mode


FinGAT(
  (short_term_gru): AttentiveGRU(
    (gru): GRU(16, 8, batch_first=True)
    (attention): Sequential(
      (0): Linear(in_features=8, out_features=8, bias=True)
      (1): Tanh()
      (2): Linear(in_features=8, out_features=1, bias=True)
    )
  )
  (stock_proj): Linear(in_features=8, out_features=8, bias=True)
  (intra_sector_transformer): DynamicTransformer(
    (layers): ModuleList(
      (0-1): 2 x DynamicTransformerLayer(
        (attention): DynamicAttention(
          (query_projection): Linear(in_features=8, out_features=8, bias=True)
          (key_projection): Linear(in_features=8, out_features=8, bias=True)
          (value_projection): Linear(in_features=8, out_features=8, bias=True)
          (output_projection): Linear(in_features=8, out_features=8, bias=True)
        )
        (layer_norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
        (layer_norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
        (feed_forward): Sequential(
        

In [11]:
for name, param in model.named_parameters():
    print(f"Layer: {name} | Shape: {param.shape}")


Layer: short_term_gru.gru.weight_ih_l0 | Shape: torch.Size([24, 16])
Layer: short_term_gru.gru.weight_hh_l0 | Shape: torch.Size([24, 8])
Layer: short_term_gru.gru.bias_ih_l0 | Shape: torch.Size([24])
Layer: short_term_gru.gru.bias_hh_l0 | Shape: torch.Size([24])
Layer: short_term_gru.attention.0.weight | Shape: torch.Size([8, 8])
Layer: short_term_gru.attention.0.bias | Shape: torch.Size([8])
Layer: short_term_gru.attention.2.weight | Shape: torch.Size([1, 8])
Layer: short_term_gru.attention.2.bias | Shape: torch.Size([1])
Layer: stock_proj.weight | Shape: torch.Size([8, 8])
Layer: stock_proj.bias | Shape: torch.Size([8])
Layer: intra_sector_transformer.layers.0.attention.query_projection.weight | Shape: torch.Size([8, 8])
Layer: intra_sector_transformer.layers.0.attention.query_projection.bias | Shape: torch.Size([8])
Layer: intra_sector_transformer.layers.0.attention.key_projection.weight | Shape: torch.Size([8, 8])
Layer: intra_sector_transformer.layers.0.attention.key_projection.bi

In [13]:
optimizer = torch.optim.Adam(model.parameters())
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 1e-05
)

In [14]:
import torch
#  Load only specific components
weights = torch.load(r"C:\Users\gargp\Desktop\NewDS\checkpoints\best_model_dim8.pt", map_location="cpu",weights_only=False)['model_state_dict']

# Inspect attention weights
print("DynamicAttention weights:", weights['intra_sector_transformer.layers.0.attention.query_projection.weight'])


DynamicAttention weights: tensor([[-0.2647,  0.0535, -0.2852,  0.0367, -0.3009,  0.0263, -0.0686, -0.1356],
        [ 0.2986,  0.1732, -0.2765, -0.2295,  0.2242,  0.1183, -0.0118, -0.2914],
        [-0.3255,  0.2772, -0.1860, -0.2745, -0.2161, -0.1527, -0.0146,  0.2745],
        [-0.0036,  0.0358,  0.3232, -0.0111,  0.3300, -0.0882, -0.0137,  0.1701],
        [ 0.1722,  0.0517, -0.2871, -0.0874, -0.0622, -0.1372,  0.2952,  0.0430],
        [-0.0981, -0.0423, -0.0906, -0.1459, -0.1129,  0.1877,  0.0739, -0.1937],
        [-0.2517, -0.2847,  0.1585,  0.0128,  0.2022, -0.0949,  0.1371,  0.2429],
        [ 0.1996, -0.1286,  0.2662,  0.1827,  0.0813, -0.2688, -0.2481, -0.1297]])


In [19]:

# Print model structure details
print("Model structure:")
for name, module in model.named_modules():
    print(f"{name}: {type(module).__name__}")

# Check if attention modules have specific attributes
intra_attention = model.intra_sector_transformer.layers[0].attention
print(f"Intra-attention attributes: {dir(intra_attention)}")


Model structure:
: FinGAT
short_term_gru: AttentiveGRU
short_term_gru.gru: GRU
short_term_gru.attention: Sequential
short_term_gru.attention.0: Linear
short_term_gru.attention.1: Tanh
short_term_gru.attention.2: Linear
stock_proj: Linear
intra_sector_transformer: DynamicTransformer
intra_sector_transformer.layers: ModuleList
intra_sector_transformer.layers.0: DynamicTransformerLayer
intra_sector_transformer.layers.0.attention: DynamicAttention
intra_sector_transformer.layers.0.attention.query_projection: Linear
intra_sector_transformer.layers.0.attention.key_projection: Linear
intra_sector_transformer.layers.0.attention.value_projection: Linear
intra_sector_transformer.layers.0.attention.output_projection: Linear
intra_sector_transformer.layers.0.layer_norm1: LayerNorm
intra_sector_transformer.layers.0.layer_norm2: LayerNorm
intra_sector_transformer.layers.0.feed_forward: Sequential
intra_sector_transformer.layers.0.feed_forward.0: Linear
intra_sector_transformer.layers.0.feed_forward.

In [1]:
import numpy as np

# Load the .npy file
data = np.load(r'C:\Users\gargp\Desktop\NewDS - Change\data\windows\360ONE_labels.npy')

# Print or inspect the contents
print(data)


[[-0.00137524  0.        ]
 [-0.05713856  0.        ]
 [ 0.03573748  1.        ]
 ...
 [-0.032276    0.        ]
 [-0.03405083  0.        ]
 [-0.00790911  0.        ]]


In [2]:
import numpy as np

# Load the .npy file
data = np.load(r'C:\Users\gargp\Desktop\NewDS - Change\data\adj_matrix.npy')

# Print or inspect the contents
print(data)


[[1. 0. 1. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [1. 0. 1. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 0. 1.]]


In [ ]:
import numpy as np

# Load the .npy file
data = np.load(r'C:\Users\gargp\Desktop\NewDS - Change\data\windows\360ONE_data.npy')

# Print or inspect the contents
print(data.shape)  #for each stock for each day,we have window of 5 and 16 features 

(736, 5, 16)


In [ ]:

import pandas as pd 
df=pd.read_csv(r'C:\Users\gargp\Desktop\NewDS - Change\data\processed\360ONE.csv');
print(df.shape);

(741, 18)
